# One Equation, All of Symmetry

## APS March Meeting 2026 — GDS Short Course on Data Science for Physicists II

**Tess Smidt** — MIT

---

Every symmetry constraint in physics — selection rules, Clebsch-Gordan coefficients, normal modes, equivariant neural networks — reduces to **one linear algebra problem**.

$$S(g) \cdot Q = Q \cdot R(g) \quad \text{for all } g \in G$$

*If you know linear algebra, you already know representation theory.*

## Setup

In [ ]:
%%capture
!pip install pymatgen 
# Install obfuscated build of symm4ml — in the course, students implement these
# functions themselves; this version lets them check answers against the reference.
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

from symm4ml import groups, linalg, rep, lie, vis, plot
from symm4ml import so3 as so3_module
from symm4ml import vib_modes
from symm4ml.utils import match_character_tables, print_character_table

np.set_printoptions(precision=3, suppress=True)

print("Ready!")

---
# Act 1: The Equation

## The fundamental problem of representation theory

Given matrices $S(g)$ and $R(g)$ for all group elements $g$, find **all** $Q$ such that:

$$S(g) \cdot Q = Q \cdot R(g) \quad \forall \, g \in G$$

This is the equation for a **similarity transform** (change of basis). When $S$ and $R$ are group representations, $Q$ is called an **intertwiner**. Every symmetry problem is an instance of it.

## It's just a nullspace problem

Rewrite $S \cdot Q - Q \cdot R = 0$ by vectorizing $Q$:

$$\bigl(S(g) \otimes I - I \otimes R(g)^\top\bigr) \, \text{vec}(Q) = 0 \quad \forall \, g$$

Stack all $g$ into one big matrix $P_\text{all}$ → find $\ker(P_\text{all})$.

That's it. **Nullspace of a matrix.** Let's look at the code.

In [ ]:
# The hero function — here's what it does under the hood:
#
# def infer_change_of_basis(S, R):
#     """Find all Q such that S @ Q = Q @ R.
#
#     For each group element g:
#         P_g = kron(S(g), I) - kron(I, R(g).T)
#
#     Stack all P_g → big matrix P_all
#     Return nullspace(P_all), reshaped into matrices
#     """
#
# That's it — Kronecker product trick + nullspace. Let's use it!

# Quick demo: does S₃'s 2D irrep admit a self-similarity?
P = groups.permutation_matrices(3)
table = groups.make_multiplication_table(P)
irreps = rep.infer_irreps(table)

for i, ir in enumerate(irreps):
    Q = linalg.infer_change_of_basis(ir, ir)
    print(f"Γ_{i} (dim {ir.shape[1]}): dim(solution space) = {len(Q)}")

## What the solution space tells you

| dim of solution space | Meaning |
|---|---|
| $\dim = 0$ | $S$ and $R$ are **inequivalent** representations |
| $\dim = 1$ (for $S = R$) | $S$ is **irreducible** (Schur's lemma!) |
| $\dim > 1$ (for $S = R$) | $S$ is **reducible** — $Q$'s eigenspaces ARE the irrep subspaces |
| Any $Q$ (for $S \neq R$) | $Q$ is a **change of basis** (intertwiner: CG coefficients, equivariant maps, ...) |

---
# Act 2: Irreps from Scratch

## Demo: The symmetric group S₃

S₃ = all permutations of 3 objects. Order 6. The symmetry group of an equilateral triangle.

Let's find **all** of its irreducible representations from scratch.

In [ ]:
# Step 1: Build S₃ from its permutation matrices
P = groups.permutation_matrices(3)  # all 3! = 6 permutation matrices
print(f"Group order: {len(P)}")
print(f"Each element is a {P[0].shape} matrix")

# Step 2: Multiplication table
table = groups.make_multiplication_table(P)
print(f"\nMultiplication table:")
print(table)

In [ ]:
# Step 3: The regular representation — 6×6 matrices
reg = rep.regular_representation(table)
print(f"Regular representation: {reg.shape}")
print(f"Each element is a {reg[0].shape} matrix")

# Is it irreducible?
print(f"\nIs the regular rep irreducible? {rep.is_an_irrep(table, reg)}")

In [ ]:
# Step 4: Find ALL self-similarity matrices Q
# reg @ Q = Q @ reg
Qs = linalg.infer_change_of_basis(reg, reg)
print(f"Number of solutions: {len(Qs)}")
print(f"Each Q has shape: {Qs[0].shape}")

# dim > 1 means the regular rep IS reducible — as expected!
# The eigenspaces of any random combination of Q's will give us the irreps.

In [ ]:
# Step 5: Take a random combination → eigenspaces → irreps!
np.random.seed(3)
coeffs = np.random.rand(len(Qs))
Q_random = np.einsum('i,ijk->jk', coeffs, Qs)

# Eigendecompose
vals, vecs = np.linalg.eig(Q_random)
eigenspaces = linalg.eigenspaces(vals, vecs)

print("Eigenvalues and their multiplicities:")
for val, space in eigenspaces:
    print(f"  λ = {val:.4f}, multiplicity = {len(space)}")

In [ ]:
# Step 6: Use the library to extract all unique irreps
irreps = rep.infer_irreps(table)

print(f"Found {len(irreps)} irreps:")
for i, irrep in enumerate(irreps):
    d = irrep.shape[1]
    print(f"  Γ_{i}: dimension {d}")

# Check: sum of dim² = |G|
dim_sq_sum = sum(ir.shape[1]**2 for ir in irreps)
print(f"\nSum of dim² = {dim_sq_sum} (should be {len(P)}) ✓" if dim_sq_sum == len(P) else f"\nSum of dim² = {dim_sq_sum} ≠ {len(P)} ✗")

### Punchline

We put in **only the multiplication table**.

We got out **all irreducible representations**.

The tool? `infer_change_of_basis` — solving $QS = RQ$.

## Molecular vibrations — water (H₂O)

C₂ᵥ symmetry, 3 atoms, 9 DOF → **3 vibrational modes**

The displacement representation = perm_rep $\otimes$ vec_rep

Let's decompose it into irreps using... $QS = RQ$.

In [ ]:
# Water molecule — C2v symmetry
# O at origin, H's symmetric about xz-plane
angle = np.radians(104.5)  # H-O-H angle
water_vertices = np.array([
    [0.0, 0.0, 0.0],                                         # O
    [np.sin(angle/2), 0.0, -np.cos(angle/2)],                # H1
    [-np.sin(angle/2), 0.0, -np.cos(angle/2)],               # H2
])
water_vertices -= water_vertices.mean(axis=0)  # center at origin

# C2v symmetry operations (pre-computed in vib_modes module)
C2v_vec = vib_modes.C2v_vec
C2v_table = vib_modes.C2v_table
C2v_irreps = list(vib_modes.C2v_irreps)

print(f"C2v has {len(C2v_vec)} symmetry operations")
print(f"All irreps are 1D: {[ir.shape[1] for ir in C2v_irreps]}")

# Match irreps to standard labels (A₁, A₂, B₁, B₂)
C2v_conj_classes = groups.conjugacy_classes(C2v_table)
C2v_char_table = rep.character_table(C2v_irreps, C2v_conj_classes)

C2v_irrep_names = ['A₁', 'A₂', 'B₁', 'B₂']
C2v_class_names = ['E', 'C₂', 'σ_v(xz)', 'σ_v(yz)']
C2v_ref_chars = np.array([
    [ 1,  1,  1,  1],   # A₁
    [ 1,  1, -1, -1],   # A₂
    [ 1, -1,  1, -1],   # B₁
    [ 1, -1, -1,  1],   # B₂
])

row_perm, col_perm = match_character_tables(C2v_char_table, C2v_ref_chars)
C2v_irreps = [C2v_irreps[i] for i in row_perm]

print_character_table(
    rep.character_table(C2v_irreps, [list(C2v_conj_classes)[j] for j in col_perm]),
    C2v_irrep_names, C2v_class_names, title="Character table of C₂ᵥ"
)

In [ ]:
# Find vibrational modes — which irreps appear?
water_modes = vib_modes.vibrational_modes(water_vertices, C2v_vec, C2v_irreps)

print("Vibrational mode decomposition:")
parts = []
for i, m in enumerate(water_modes):
    if m.shape[0] > 0:
        dim = C2v_irreps[i].shape[1]
        n_copies = m.shape[0] // dim
        label = C2v_irrep_names[i]
        print(f"  {label} (dim={dim}): {n_copies} copy/copies")
        parts.append(f"{n_copies}{label}" if n_copies > 1 else label)

print(f"\nΓ_vib = {' + '.join(parts)}")
print(f"\n→ 2 symmetric modes (A₁: stretch + bend) and 1 asymmetric stretch (B₁)")

In [ ]:
# Projector decomposition: total DOF = translation + rotation + vibration
fig = plot.plot_projectors(water_vertices)
plt.show()

In [ ]:
# Animated vibrational mode viewer
HTML(plot.vibrational_modes_viewer(
    water_vertices,
    water_modes,
    irrep_labels=C2v_irrep_names,
    bonds=[[0, 1], [0, 2]],
    atom_colors=["#e53935", "#1565c0", "#00897b"],  # O=red, H1=blue, H2=teal
    atom_sizes=[16, 12, 12],
    atom_labels=["O", "H", "H"],
    scale=0.3,
))

### Punchline

The **same equation** that found abstract irreps now tells us **which atoms move together**.

Out come the symmetric stretch ($A_1$), bend ($A_1$), and asymmetric stretch ($B_1$) — all from $QS = RQ$.

---
# Act 3: SO(3) — The Big One

## From finite groups to continuous groups

For a Lie group, we can't enumerate all $g$. But we can work with the **generators** (Lie algebra).

Same equation, same code:

$$X_\text{out}^{(a)} \cdot Q = Q \cdot X_\text{in}^{(a)} \quad \forall \, a = 1, \ldots, \dim(\mathfrak{g})$$

For SO(3): three generators $X^{(a)}$ = angular momentum operators $L_x, L_y, L_z$.

In [ ]:
# SO(3) generators — the l=1 angular momentum matrices
X = lie.so3_X  # shape (3, 3, 3)
print("SO(3) generators (angular momentum operators):")
labels = ['Lx', 'Ly', 'Lz']
for label, gen in zip(labels, X):
    print(f"\n{label} =")
    print(gen)

## Discovering spherical harmonics

**Key idea:** Tensor product two representations → decompose into irreps → discover new irreps.

$l=1 \otimes l=1 = l=0 \oplus l=1 \oplus l=2$

Let's derive this computationally.

In [ ]:
# Tensor product of l=1 with itself
X_1x1 = lie.tensor_product(X, X)  # 9×9 generators
print(f"l=1 ⊗ l=1 generators: shape {X_1x1.shape}")

# Decompose into irreps
irreps_from_tp = lie.decompose_rep_into_irreps(X_1x1)

print(f"\nDecomposition of l=1 ⊗ l=1:")
for ir in irreps_from_tp:
    l = (ir.shape[1] - 1) // 2
    print(f"  l = {l} (dim = {ir.shape[1]})")

print(f"\n✓ We just derived 1 ⊗ 1 = 0 ⊕ 1 ⊕ 2 computationally!")

In [ ]:
# Build ALL irreps up to l=5 by recursive tensor products
Xs = lie.infer_irreps_from_tensor_products(X, 6)  # l=0 through l=5

print("All SO(3) irreps found:")
for i, ir in enumerate(Xs):
    l = (ir.shape[1] - 1) // 2
    print(f"  l = {l}: generators shape {ir.shape}")

print(f"\nWe started with l=1 (angular momentum).")
print(f"We got ALL irreps up to l=5 — the spherical harmonics!")

In [ ]:
# Evaluate spherical harmonics at a point
x = np.array([1.0, 0.5, -0.3])
Ys = so3_module.spherical_harmonics(Xs, x)

print(f"Spherical harmonics Y_l^m at x = {x}:")
for l, Y in enumerate(Ys):
    print(f"  l={l}: {Y}")

In [ ]:
# Visualize spherical harmonics on a sphere
theta = np.linspace(0, np.pi, 60)
phi = np.linspace(0, 2*np.pi, 120)
THETA, PHI = np.meshgrid(theta, phi)

# Convert to Cartesian
x_grid = np.sin(THETA) * np.cos(PHI)
y_grid = np.sin(THETA) * np.sin(PHI)
z_grid = np.cos(THETA)

points = np.stack([x_grid, y_grid, z_grid], axis=-1)  # (120, 60, 3)

# Evaluate Y_l^m for l=0,1,2
Xs_3 = lie.infer_irreps_from_tensor_products(X, 3)
Ys_grid = so3_module.spherical_harmonics(Xs_3, points)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), subplot_kw={'projection': '3d'})

for l, ax in enumerate(axes):
    # Pick the m=0 component (middle index)
    m_idx = l  # m=0 is at index l
    Y_vals = Ys_grid[l][..., m_idx]

    # Surface colored by Y value
    r = np.abs(Y_vals)
    ax.plot_surface(r*x_grid, r*y_grid, r*z_grid,
                    facecolors=plt.cm.coolwarm((Y_vals - Y_vals.min()) / (Y_vals.max() - Y_vals.min() + 1e-10)),
                    alpha=0.8)
    ax.set_title(f'$Y_{l}^0$', fontsize=16)
    ax.set_box_aspect([1,1,1])
    ax.axis('off')

plt.suptitle('Spherical Harmonics (m=0) — discovered via QS = RQ', fontsize=14)
plt.tight_layout()
plt.show()

## Clebsch-Gordan coefficients

The CG coefficients you look up in textbooks are **solutions to** $QS = RQ$.

$$D^{(l_1)}(g) \otimes D^{(l_2)}(g) \cdot Q = Q \cdot D^{(l_3)}(g)$$

The $Q$ that solves this IS the CG matrix.

In [ ]:
# Clebsch-Gordan coefficients: l=1 ⊗ l=1 → l=2
CG = so3_module.clebsch_gordan(Xs[1], Xs[1], Xs[2])
print(f"CG(l=1, l=1 → l=2) shape: {CG.shape}")
print(f"This IS the Q from (l=1 ⊗ l=1) @ Q = Q @ l=2\n")

# Show the CG coefficients
print("CG coefficients (3 × 3 × 5 tensor):")
for m3 in range(5):
    print(f"\n  m₃ = {m3-2}:")
    print(CG[:, :, m3].round(3))

In [ ]:
# CG for l=1 ⊗ l=1 → l=0 (the scalar/dot product)
CG_00 = so3_module.clebsch_gordan(Xs[1], Xs[1], Xs[0])
print(f"CG(l=1, l=1 → l=0) shape: {CG_00.shape}")
print(f"\nCG matrix (should be proportional to identity):")
print(CG_00[:, :, 0].round(3))
print(f"\nThis is the dot product! l=1 ⊗ l=1 → l=0 extracts the scalar part.")

## Equivariant linear maps

An equivariant linear map $W$ must satisfy:

$$D_\text{out}(g) \cdot W = W \cdot D_\text{in}(g) \quad \forall \, g$$

This IS $QS = RQ$ with $Q = W$.

The solution space = **the allowed weight matrices** in an equivariant neural network.

In [ ]:
# How many free parameters for l=1 → l=1?
W_1to1 = linalg.infer_change_of_basis(Xs[1], Xs[1])
print(f"l=1 → l=1: {len(W_1to1)} free parameter(s)")
print(f"The solution is proportional to the identity — Schur's lemma!")
print(f"W =\n{W_1to1[0].round(3)}")

In [ ]:
# For l=0 ⊕ l=1 → l=0 ⊕ l=1?
X_01 = np.zeros((3, 4, 4))  # direct sum of l=0 (1×1) and l=1 (3×3)
X_01[:, 1:, 1:] = Xs[1]     # l=1 block in bottom-right

W_01 = linalg.infer_change_of_basis(X_01, X_01)
print(f"l=0 ⊕ l=1 → l=0 ⊕ l=1: {len(W_01)} free parameter(s)")
print(f"\nThe block structure emerges automatically:")
for i, w in enumerate(W_01):
    print(f"\nW_{i} =")
    print(w.round(3))

print(f"\n→ l=0 and l=1 blocks are independent. No mixing between different l!")

In [ ]:
# Can we map l=0 → l=1? (scalar to vector)
W_0to1 = linalg.infer_change_of_basis(Xs[1], Xs[0])
print(f"l=0 → l=1: {len(W_0to1)} free parameter(s)")
print(f"\nNo equivariant linear map from scalars to vectors exists!")
print(f"You can't build a vector from a scalar without breaking symmetry.")

---
# Act 4: Equivariant Machine Learning

## Architecture of equivariant GNNs

Models like **MACE**, **NequIP**, and **Allegro** use:
- **Tensor product layers** → CG coefficients = solutions to $QS = RQ$
- **Linear layers** → weight space constrained by $QS = RQ$

Fewer parameters → better data efficiency → why these work for materials.

**Key insight**: We only need to build in the symmetry of **empty space** (O(3) / E(3)). The data itself lowers the symmetry — a single equivariant architecture handles any molecule, any crystal, any symmetry.

### Standard NN vs. Equivariant NN

| | Standard NN | Equivariant NN |
|---|---|---|
| Weight matrix | Free $d_\text{out} \times d_\text{in}$ matrix | Lives in nullspace of $QS = RQ$ |
| Parameters | $d_\text{out} \cdot d_\text{in}$ | Much fewer (10-100× reduction for SO(3)) |
| Data needed | More | Less (symmetry = free data augmentation) |

In [ ]:
# Parameter comparison: standard vs equivariant
# Input: 2×l=0 + 2×l=1 + 1×l=2 = 2 + 6 + 5 = 13 dimensions
# Output: same

# Build input/output reps
reps_to_sum = [Xs[0], Xs[0], Xs[1], Xs[1], Xs[2]]  # 2×l=0 + 2×l=1 + 1×l=2
D_in = rep.direct_sum_multiple(reps_to_sum)

d = D_in.shape[1]
print(f"Feature dimension: {d}")
print(f"Standard NN parameters: {d*d} = {d}×{d}")

# Equivariant: count solutions to D @ W = W @ D
W_equivariant = linalg.infer_change_of_basis(D_in, D_in)
print(f"Equivariant NN parameters: {len(W_equivariant)}")
print(f"\nReduction: {d*d}/{len(W_equivariant)} = {d*d/len(W_equivariant):.1f}× fewer parameters!")

## Crystal field theory from $QS = RQ$

How do atomic d-orbitals ($l=2$, 5-fold degenerate) split under octahedral symmetry ($O_h$)?

Restrict the $l=2$ representation of SO(3) to the octahedral subgroup → decompose using $QS = RQ$.

Out come $t_{2g}$ and $e_g$ — **we just derive crystal field theory!**

In [ ]:
import scipy.linalg

# Get octahedral point group operations (3×3 matrices)
Oh_ops = vib_modes.point_group_ops('Oh')
print(f"Oh has {len(Oh_ops)} symmetry operations")

# Build the l=2 representation of each Oh operation
# D^(l=2)(R) = exp(angle * axis · X^(l=2))
X2 = Xs[2]  # l=2 generators, shape (3, 5, 5)

D2_Oh = np.zeros((len(Oh_ops), 5, 5))
for i, R in enumerate(Oh_ops):
    # Handle improper rotations: det(R) = -1 means inversion × proper rotation
    det = np.linalg.det(R)
    R_proper = R * np.sign(det)  # strip inversion if present
    
    # Get rotation angle and axis
    angle = so3_module.rotation_angle_from_matrix(R_proper)
    if angle < 1e-10:
        D2_Oh[i] = np.eye(5)
    else:
        # Find rotation axis from antisymmetric part
        axis = np.array([R_proper[2,1]-R_proper[1,2],
                         R_proper[0,2]-R_proper[2,0],
                         R_proper[1,0]-R_proper[0,1]])
        axis_norm = np.linalg.norm(axis)
        if axis_norm < 1e-10:
            # 180° rotation — axis is eigenvector with eigenvalue +1
            vals, vecs = np.linalg.eig(R_proper)
            idx = np.argmin(np.abs(vals - 1.0))
            axis = np.real(vecs[:, idx])
        else:
            axis = axis / axis_norm
        
        # D^(2)(R) = exp(angle * axis · X^(2))
        log_coords = angle * axis
        D2_Oh[i] = scipy.linalg.expm(np.einsum('aij,a->ij', X2, log_coords))
    
    # For improper rotations: d-orbitals are even parity (l=2), so D(-R) = D(R)
    # (parity = (-1)^l = +1 for l=2)

print(f"\nD^(l=2) restricted to Oh: shape {D2_Oh.shape}")

In [ ]:
# Decompose D^(2) restricted to Oh into irreps
Oh_table = groups.make_multiplication_table(Oh_ops)
Oh_irreps = rep.infer_irreps(Oh_table)

print(f"Oh has {len(Oh_irreps)} irreps with dimensions: {[ir.shape[1] for ir in Oh_irreps]}")

# Find which irreps appear in the l=2 restriction
d2_irreps = rep.decompose_rep_into_irreps(D2_Oh)
print(f"\nl=2 under Oh decomposes into:")
for ir in d2_irreps:
    print(f"  dim {ir.shape[1]} irrep")

print(f"\n→ d-orbitals split into: t₂g (3-fold) + eg (2-fold)")
print(f"  This is crystal field theory!")

## Connection to generative AI for materials

Generative models for crystals (DiffCSP, CDVAE, FlowMM) use **equivariant architectures**:

- The score function / denoising network must be equivariant
- Same $QS = RQ$ equation constrains the network weights
- Generated structures automatically respect rotational symmetry

The math is the same whether you're doing spectroscopy, phonons, or training a neural network.

---
# Act 5: The Unifying Picture

## One equation, many applications

| Application | $S$ | $R$ | $Q$ = ? |
|---|---|---|---|
| Rep equivalence | rep₁ | rep₂ | change of basis |
| Schur's lemma | irrep | irrep | $\propto$ identity |
| Irrep decomposition | rep | rep | eigenspaces = irreps |
| CG coefficients | $l_1 \otimes l_2$ | $l_3$ | CG matrix |
| Equivariant weights | $D_\text{out}$ | $D_\text{in}$ | allowed $W$ matrices |
| Spherical harmonics | $D^{(l+1)}$ | $D^{(l)} \otimes D^{(1)}$ | $Y^{(l+1)}$ projection |
| Normal modes | vib rep | irrep | mode shapes |
| Crystal field | SO(3) irrep | point group irrep | orbital splitting |
| Convention matching | $Y_{lm}$ (code A) | $Y_{lm}$ (code B) | basis transform |

## Call to action

**You already know linear algebra. Representation theory is just knowing which matrices to plug into $S$ and $R$.**

### Resources

- **symm4ml** — the Python package built by students in 6.7970/8.750, used in this tutorial. The version installed above is an obfuscated build — in the course, students implement the functions themselves for homework.
- **e3nn** — equivariant neural network library: [e3nn.org](https://e3nn.org)
- **6.7970/8.750** — Full MIT course: *Symmetry and its Application to Machine Learning*

### The takeaway

Every symmetry constraint — selection rules, CG coefficients, normal modes, equivariant neural networks — is an instance of:

$$\boxed{S(g) \cdot Q = Q \cdot R(g)}$$

One equation. All of symmetry.

In [ ]:
# Final demo: let's verify the summary table row by row

print("=" * 60)
print("VERIFICATION: Every row of the summary table")
print("=" * 60)

# 1. Rep equivalence
S3_table = groups.make_multiplication_table(groups.permutation_matrices(3))
S3_irreps = rep.infer_irreps(S3_table)
Q_equiv = linalg.infer_change_of_basis(S3_irreps[0], S3_irreps[0])
print(f"\n1. Rep equivalence: dim(Q) = {len(Q_equiv)} (equivalent reps)")

# 2. Schur's lemma
Q_schur = linalg.infer_change_of_basis(S3_irreps[-1], S3_irreps[-1])  # 2D irrep
print(f"2. Schur's lemma: dim(Q) = {len(Q_schur)} for irrep (∝ identity) ✓")

# 3. Irrep decomposition
S3_reg = rep.regular_representation(S3_table)
Q_decomp = linalg.infer_change_of_basis(S3_reg, S3_reg)
print(f"3. Irrep decomposition: dim(Q) = {len(Q_decomp)} for regular rep")

# 4. CG coefficients
CG_12 = so3_module.clebsch_gordan(Xs[1], Xs[1], Xs[2])
print(f"4. CG coefficients: shape {CG_12.shape} for l=1⊗l=1→l=2")

# 5. Equivariant weights
W = linalg.infer_change_of_basis(Xs[2], Xs[2])
print(f"5. Equivariant weights: {len(W)} free param(s) for l=2→l=2")

# 6. Inequivalent reps
Q_ineq = linalg.infer_change_of_basis(Xs[1], Xs[2])
print(f"6. l=1 ≠ l=2: dim(Q) = {len(Q_ineq)} (inequivalent!)")

print(f"\n" + "=" * 60)
print("All verified. One equation, all of symmetry. ✓")
print("=" * 60)